# Simulación de máquinas de Turing

Este cuaderno implementa un simulador de máquina de Turing determinista y lo usa para estudiar tres ejemplos concretos: reconocimiento de palíndromos, verificación de paridad y suma unaria.

El objetivo es hacer observable el comportamiento paso a paso de una MT: cinta, cabezal y estado en cada transición.

**Artículo asociado:** [Máquinas de Turing](../../03-computabilidad/04-maquinas-de-turing.md)

## El modelo

Una máquina de Turing determinista queda definida por:
- Un conjunto de **estados** $Q$ con estado inicial $q_0$ y estados de aceptación $F$.
- Un **alfabeto de cinta** $\Gamma$ que incluye el símbolo en blanco $B$.
- Una **función de transición** $\delta: Q \times \Gamma \to Q \times \Gamma \times \{L, R\}$.

En cada paso: leer el símbolo bajo el cabezal, consultar $\delta$, escribir un símbolo, mover el cabezal y cambiar de estado.

In [ ]:
class TuringMachine:
    """Simulador de máquina de Turing determinista con cinta infinita."""

    BLANK = '_'

    def __init__(self, states, initial, accepting, rejecting, transitions):
        """
        states      : iterable de estados
        initial     : estado inicial
        accepting   : estado de aceptación
        rejecting   : estado de rechazo
        transitions : dict {(estado, símbolo): (nuevo_estado, símbolo_escribir, dirección)}
                      dirección en {'L', 'R'}
        """
        self.states = set(states)
        self.initial = initial
        self.accepting = accepting
        self.rejecting = rejecting
        self.transitions = transitions

    def run(self, input_string, max_steps=1000, verbose=False):
        """Ejecuta la MT sobre input_string. Devuelve True si acepta."""
        tape = list(input_string) if input_string else [self.BLANK]
        head = 0
        state = self.initial
        steps = 0

        while steps < max_steps:
            # Extender la cinta si el cabezal sale por la derecha
            if head >= len(tape):
                tape.append(self.BLANK)
            if head < 0:
                tape.insert(0, self.BLANK)
                head = 0

            symbol = tape[head]

            if verbose:
                self._print_config(tape, head, state, steps)

            if state == self.accepting:
                return True
            if state == self.rejecting:
                return False

            key = (state, symbol)
            if key not in self.transitions:
                # Sin transición definida: rechazar
                return False

            new_state, write_symbol, direction = self.transitions[key]
            tape[head] = write_symbol
            state = new_state
            head += 1 if direction == 'R' else -1
            steps += 1

        raise RuntimeError(f'La MT no terminó en {max_steps} pasos.')

    def _print_config(self, tape, head, state, step):
        tape_str = ''.join(tape)
        pointer = ' ' * head + '^'
        print(f'Paso {step:3d} | Estado: {state:10s} | Cinta: {tape_str}')
        print(f'           |           |        {pointer}')

print('Simulador de MT listo.')

## Ejemplo 1: reconocer palíndromos binarios

Reconocemos el lenguaje $\{w \in \{0,1\}^* : w = w^R\}$.

**Estrategia:** comparar el primer y último símbolo, borrarlos, y repetir hasta que la cinta quede vacía o con un solo símbolo.

In [ ]:
# MT para palíndromos binarios
# Estados:
#   q0: leer el primer símbolo y memorizarlo
#   q1_seek_end_0, q1_seek_end_1: buscar el extremo derecho recordando '0' o '1'
#   q2_check_0, q2_check_1: comparar el extremo derecho con lo memorizado
#   q3: volver al extremo izquierdo
#   qa: aceptar, qr: rechazar

transitions_palindrome = {
    # Desde q0: leer el primer símbolo
    ('q0', '0'): ('q1_0', '_', 'R'),   # borrar '0', buscar el final
    ('q0', '1'): ('q1_1', '_', 'R'),   # borrar '1', buscar el final
    ('q0', '_'): ('qa',   '_', 'R'),   # cinta vacía o quedó un símbolo: aceptar

    # Desde q1_0 y q1_1: moverse hasta el final
    ('q1_0', '0'): ('q1_0', '0', 'R'),
    ('q1_0', '1'): ('q1_0', '1', 'R'),
    ('q1_0', '_'): ('q2_0', '_', 'L'),  # llegamos al final: retroceder y comparar
    ('q1_1', '0'): ('q1_1', '0', 'R'),
    ('q1_1', '1'): ('q1_1', '1', 'R'),
    ('q1_1', '_'): ('q2_1', '_', 'L'),

    # Desde q2_0: comparar el último símbolo con '0'
    ('q2_0', '0'): ('q3', '_', 'L'),   # coincide: borrar y volver
    ('q2_0', '1'): ('qr', '1', 'L'),   # no coincide: rechazar
    ('q2_0', '_'): ('qa', '_', 'R'),   # solo quedaba el blanco (longitud impar ok)

    # Desde q2_1: comparar el último símbolo con '1'
    ('q2_1', '1'): ('q3', '_', 'L'),
    ('q2_1', '0'): ('qr', '0', 'L'),
    ('q2_1', '_'): ('qa', '_', 'R'),

    # Desde q3: volver al inicio (primer blanco a la izquierda)
    ('q3', '0'): ('q3', '0', 'L'),
    ('q3', '1'): ('q3', '1', 'L'),
    ('q3', '_'): ('q0', '_', 'R'),     # llegamos al blanco inicial: reiniciar
}

mt_palindrome = TuringMachine(
    states={'q0','q1_0','q1_1','q2_0','q2_1','q3','qa','qr'},
    initial='q0',
    accepting='qa',
    rejecting='qr',
    transitions=transitions_palindrome
)

casos = ['', '0', '1', '00', '01', '10', '11', '010', '001',
         '0110', '0100', '10101', '10001']

print('Palíndromos binarios:')
print(f'{"Entrada":>10}  {"Acepta"}')
for w in casos:
    resultado = mt_palindrome.run(w)
    print(f'{repr(w):>10}  {resultado}')

### Traza paso a paso para `'010'`

In [ ]:
print('Traza para "010":')
mt_palindrome.run('010', verbose=True)

## Ejemplo 2: paridad de una cadena binaria

Reconocemos el lenguaje de cadenas binarias con número par de unos.

**Estrategia:** un estado registra si hemos visto un número par o impar de unos hasta el momento. Cuando leemos el blanco final, aceptamos si el contador es 'par'.

In [ ]:
transitions_parity = {
    ('par',  '0'): ('par',  '0', 'R'),
    ('par',  '1'): ('impar','1', 'R'),
    ('impar','0'): ('impar','0', 'R'),
    ('impar','1'): ('par',  '1', 'R'),
    ('par',  '_'): ('qa',   '_', 'R'),  # número de unos par: aceptar
    ('impar','_'): ('qr',   '_', 'R'),  # número de unos impar: rechazar
}

mt_parity = TuringMachine(
    states={'par','impar','qa','qr'},
    initial='par',
    accepting='qa',
    rejecting='qr',
    transitions=transitions_parity
)

casos_paridad = ['', '0', '1', '00', '01', '10', '11', '110', '1101', '10101']
print('Paridad (número par de unos):')
print(f'{"Entrada":>10}  {"Unos":>4}  {"Acepta"}')
for w in casos_paridad:
    resultado = mt_parity.run(w)
    unos = w.count('1')
    print(f'{repr(w):>10}  {unos:>4}  {resultado}')

## Ejemplo 3: incrementar un número unario

En representación unaria, el número $n$ se escribe como $n$ palitos: `|` × n.

Esta MT añade un palito al final: transforma $n$ en $n+1$.

In [ ]:
transitions_increment = {
    ('q0', '|'): ('q0', '|', 'R'),   # avanzar hasta el final
    ('q0', '_'): ('qa', '|', 'R'),   # añadir un palito al llegar al blanco
}

mt_increment = TuringMachine(
    states={'q0','qa'},
    initial='q0',
    accepting='qa',
    rejecting='qr',   # nunca se llega a este estado
    transitions=transitions_increment
)

def unario(n):
    return '|' * n

def contar_palitos(tape_result, input_str):
    # Ejecutar la MT y contar palitos en la cinta
    tape = list(input_str) if input_str else ['_']
    head = 0
    state = 'q0'
    for _ in range(200):
        if head >= len(tape):
            tape.append('_')
        symbol = tape[head]
        if state == 'qa':
            break
        key = (state, symbol)
        if key not in transitions_increment:
            break
        new_state, write, direction = transitions_increment[key]
        tape[head] = write
        state = new_state
        head += 1 if direction == 'R' else -1
    return ''.join(tape).strip('_').count('|')

print('Incremento unario:')
print(f'{"Entrada (n)":>12}  {"Resultado (n+1)"}')
for n in range(6):
    entrada = unario(n)
    resultado = contar_palitos(None, entrada)
    print(f'{n:>12}  {resultado}')

## Análisis de complejidad

¿Cuántos pasos necesita cada MT en función de la longitud $n$ de la entrada?

In [ ]:
import math

def contar_pasos(mt, input_string, max_steps=5000):
    """Cuenta los pasos que necesita la MT para terminar."""
    tape = list(input_string) if input_string else [mt.BLANK]
    head = 0
    state = mt.initial
    steps = 0
    while steps < max_steps:
        if head >= len(tape):
            tape.append(mt.BLANK)
        if head < 0:
            tape.insert(0, mt.BLANK)
            head = 0
        symbol = tape[head]
        if state in (mt.accepting, mt.rejecting):
            return steps
        key = (state, symbol)
        if key not in mt.transitions:
            return steps
        new_state, write_symbol, direction = mt.transitions[key]
        tape[head] = write_symbol
        state = new_state
        head += 1 if direction == 'R' else -1
        steps += 1
    return steps

# Palíndromos: cadenas del tipo 0^n 1 0^n (longitud 2n+1)
print('Pasos para reconocer palíndromos 0^n 1 0^n:')
print(f'{"n":>4}  {"longitud":>8}  {"pasos":>8}')
for n in range(1, 9):
    w = '0' * n + '1' + '0' * n
    pasos = contar_pasos(mt_palindrome, w)
    print(f'{n:>4}  {len(w):>8}  {pasos:>8}')

print()
print('Pasos para verificar paridad de 1^n:')
print(f'{"n":>4}  {"longitud":>8}  {"pasos":>8}')
for n in range(1, 9):
    w = '1' * n
    pasos = contar_pasos(mt_parity, w)
    print(f'{n:>4}  {len(w):>8}  {pasos:>8}')

## Observaciones

La MT de paridad necesita exactamente $n+1$ pasos para una cadena de longitud $n$: recorre la cinta una sola vez de izquierda a derecha. Es **tiempo lineal**.

La MT de palíndromos necesita aproximadamente $n^2$ pasos: en cada iteración recorre la cinta completa para comparar un par de símbolos. Es **tiempo cuadrático** con esta estrategia (aunque existe una MT de 2 cintas que lo hace en tiempo lineal).

Esto ilustra que la complejidad temporal depende no solo del problema sino también del diseño de la MT.